## В данной задаче делаю предсказание возраста опоссума

### Загружаю датасет с https://www.kaggle.com/datasets/abrambeyer/openintro-possum

In [1]:
pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

dataset_name = "abrambeyer/openintro-possum"
save_path = os.path.expanduser("~/Py/DLS_mipt/PossumRegression/kaggle_data")

os.makedirs(save_path, exist_ok=True)
os.system(f"kaggle datasets download -d {dataset_name} -p {save_path} --unzip")


Dataset URL: https://www.kaggle.com/datasets/abrambeyer/openintro-possum
License(s): CC0-1.0


0

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
df = pd.read_csv("kaggle_data/possum.csv")
df.head()

/home/morrowto/.local/lib/python3.10/site-packages/numpy/_core/getlimits.py:551: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)


,case,site,Pop,sex,age,hdlngth,skullw,totlngth,taill,footlgth,earconch,eye,chest,belly
0,1,1,Vic,m,8.0,94.1,60.4,89.0,36.0,74.5,54.5,15.2,28.0,36.0
1,2,1,Vic,f,6.0,92.5,57.6,91.5,36.5,72.5,51.2,16.0,28.5,33.0
2,3,1,Vic,f,6.0,94.0,60.0,95.5,39.0,75.4,51.9,15.5,30.0,34.0
3,4,1,Vic,f,6.0,93.2,57.1,92.0,38.0,76.1,52.2,15.2,28.0,34.0
4,5,1,Vic,f,2.0,91.5,56.3,85.5,36.0,71.0,53.2,15.1,28.5,33.0


### Подготовка датасета
#### site, Pop, case, sex - допускаю, что влияния нет

In [4]:
df.drop(columns=['site', 'Pop', 'case', 'sex'], inplace = True)

In [5]:
df.isna().sum()

age         2
hdlngth     0
skullw      0
totlngth    0
taill       0
footlgth    1
earconch    0
eye         0
chest       0
belly       0
dtype: int64

In [6]:
#удаляем Nan
df.dropna(inplace=True)

In [7]:
X = df.drop(columns=['age']).values
y = df['age'].values

In [8]:
!pip uninstall -y scikit-learn # удалим более старую версию библиотеки
!pip install scikit-learn

Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Defaulting to user installation because normal site-packages is not writeable
  Using cached scikit_learn-1.6.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (13.5 MB)


In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, test_size=0.2, random_state=44)

### kNN алгоритм

In [10]:
from math import sqrt
def calc_distance(vec1, vec2):
    distance = 0.0
    for i in range(len(vec1)):
        distance += (vec1[i] - vec2[i])**2
    return sqrt(distance)

In [11]:
calc_distance(X_train[0], X_test[0])

15.320574401764446

In [12]:
def get_k_neighbour(train, test_row, num_neighbour):
    distances = []
    nearest_neighbour_ids = []
    for train_id, train_row in enumerate(train):
        distance_train_and_test = calc_distance(train_row, test_row)
        distances.append((train_id, distance_train_and_test))

    distances.sort(key=lambda x: x[1])
    for i in range(num_neighbour):
        nearest_neighbour_ids.append(distances[i][0])
    return nearest_neighbour_ids

In [13]:
get_k_neighbour(X_train[:5], X_test[1], 3)

[4, 2, 3]

In [14]:
def predict(X_train, X_test, y_train, num_neighbour = 3):
    y_predict = []
    for x_test in X_test:
        nearest_neighbour_ids = get_k_neighbour(X_train, x_test, num_neighbour)
        y_preds = y_train[nearest_neighbour_ids]
        y_preds = y_preds.mean()
        y_predict.append(y_preds)
    return y_predict

In [15]:
y_predict = predict(X_train[:30], X_test[:5], y_train[:30], num_neighbour = 5)
y_predict

[np.float64(4.4),
 np.float64(3.4),
 np.float64(2.8),
 np.float64(3.4),
 np.float64(3.4)]

### kNN в Sklearn

In [16]:
from sklearn.neighbors import KNeighborsRegressor

model = KNeighborsRegressor(n_neighbors = 5)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred

array([4.4, 4. , 3.2, 5.8, 4. , 4. , 4.6, 2.4, 4.6, 3.8, 2. , 5. , 3. ,
       5.2, 5.8, 5. , 2.2, 2.8, 4.8, 1.6, 3. ])